# EDA — Part 1: Univariate & Multivariate Exploration

**Module 9: Exploratory Data Analysis (EDA)**

## Learning Objectives
- Understand the difference between *knowing how to use pandas* and *doing EDA with pandas*
- Build a systematic univariate profile of every variable in a dataset
- Use `.groupby()`, crosstabs and pivot tables as analytical tools, not just aggregation tools
- Generate concrete, testable hypotheses before producing a single chart
- Know what questions to ask at the start of any EDA

## Context
> *"The greatest value of a picture is when it forces us to notice what we never expected to see."*
> — John Tukey, inventor of EDA

In **Modules 3 and 4** you learned *how* to use pandas and statistics. In **EDA Part 1**, the goal is different: we use those same tools with a *questioning mindset*. Every operation you run should be answering — or generating — a question about the data.

Parts 1, 2 and 3 form a complete EDA workflow:

| Part | Focus | Tools |
|------|-------|-------|
| **Part 1 (this notebook)** | Understand your data structurally before visualising | `.describe()`, `.value_counts()`, `.groupby()`, crosstabs, pivot tables |
| **Part 2** | Visualise distributions and relationships | Histograms, boxplots, pairplots, heatmaps |
| **Part 3** | Track changes over time and quantify relationships | Time series, rolling averages, correlation |

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✓ Libraries loaded successfully')

✓ Libraries loaded successfully


In [2]:
# Load the Superstore dataset — same dataset used in Modules 6, 9 Part 2 and 9 Part 3
# Module 5 reference: datetime parsing → 03_type_conversion_datetime.ipynb

from pathlib import Path

data_path = Path('Superstore.csv')

df = pd.read_csv(data_path, encoding='latin1')

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print(f'Shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head(3)

Shape: (9994, 21)
Memory usage: 8450.1 KB


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87


---
## 1. The EDA Mindset: Questions Before Answers

When you load a new dataset, resist the temptation to run `.describe()` and immediately start plotting. Instead, ask structured questions first:

### The four questions you always ask first

| Question | What you're looking for |
|----------|------------------------|
| **What is the unit of observation?** | What does one row represent? (a transaction? a customer? a day?) |
| **What is the time span and granularity?** | How many years? Daily, monthly, yearly data? |
| **What types of variables do I have?** | Numerical, categorical, datetime, free text? |
| **What is the data quality?** | Missing values, duplicates, inconsistent categories, wrong types? |

These questions shape everything that follows. Let us answer them for Superstore.

In [3]:
# Q1: What is the unit of observation?
print('=== Unit of observation ===')
print(f'Total rows: {len(df):,}')
print(f'Unique Order IDs: {df["Order ID"].nunique():,}')
print(f'Unique Row IDs: {df["Row ID"].nunique():,}')
print()
# Are rows unique?
print(f'Duplicate rows: {df.duplicated().sum()}')
# Multiple rows per order?
rows_per_order = df.groupby('Order ID').size()
print(f'Average rows per Order ID: {rows_per_order.mean():.1f}')
print(f'Max rows per Order ID: {rows_per_order.max()}')

=== Unit of observation ===
Total rows: 9,994
Unique Order IDs: 5,009
Unique Row IDs: 9,994

Duplicate rows: 0
Average rows per Order ID: 2.0
Max rows per Order ID: 14


### Analysis

Each row is **one order line** (a product within an order), not one order. A single Order ID can appear in multiple rows — one per product purchased. This is a critical finding: when we compute totals per order, we need to `groupby('Order ID')`, not assume one row = one order.

Understanding the unit of observation prevents errors like double-counting customers or orders.

In [4]:
# Q2: Time span and granularity
print('=== Time span ===')
print(f'Date range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Years covered: {sorted(df["Order Date"].dt.year.unique())}')
print(f'Total months with data: {df["Order Date"].dt.to_period("M").nunique()}')
print()
# Q3: Variable types
print('=== Variable types ===')
print(df.dtypes.value_counts())
print()
print(df.dtypes)

=== Time span ===
Date range: 2014-01-03 → 2017-12-30
Years covered: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]
Total months with data: 48

=== Variable types ===
object            13
int64              3
float64            3
datetime64[ns]     2
Name: count, dtype: int64

Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code               int64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object


In [5]:
# Q4: Data quality overview
# Module 5 reference: 01_dealing_with_missing_data.ipynb

print('=== Data quality snapshot ===')
quality = pd.DataFrame({
    'dtype':        df.dtypes,
    'non_null':     df.count(),
    'missing':      df.isnull().sum(),
    'missing_%':    (df.isnull().mean() * 100).round(1),
    'unique':       df.nunique(),
    'unique_%':     (df.nunique() / len(df) * 100).round(1),
})
print(quality.to_string())

=== Data quality snapshot ===
                        dtype  non_null  missing  missing_%  unique  unique_%
Row ID                  int64      9994        0       0.00    9994    100.00
Order ID               object      9994        0       0.00    5009     50.10
Order Date     datetime64[ns]      9994        0       0.00    1237     12.40
Ship Date      datetime64[ns]      9994        0       0.00    1334     13.30
Ship Mode              object      9994        0       0.00       4      0.00
Customer ID            object      9994        0       0.00     793      7.90
Customer Name          object      9994        0       0.00     793      7.90
Segment                object      9994        0       0.00       3      0.00
Country                object      9994        0       0.00       1      0.00
City                   object      9994        0       0.00     531      5.30
State                  object      9994        0       0.00      49      0.50
Postal Code             int64     

### Analysis

This quality snapshot is one of the most useful tables in any EDA. At a glance it tells us:

- **Missing values**: Superstore has none — unusual for real-world data, but good for a learning dataset.
- **High cardinality columns** (unique_% close to 100%): `Order ID`, `Customer Name`, `Product Name` — these are identifiers, not categories to group by blindly.
- **Low cardinality columns** (few unique values): `Category`, `Region`, `Segment`, `Ship Mode` — these are the natural grouping variables.

This distinction — high vs low cardinality — is fundamental to deciding **which columns are useful for groupby operations**.

---
## 2. Univariate Exploration: One Variable at a Time

Univariate exploration means studying each variable **in isolation** before looking at relationships.
The goal is to understand the *shape*, *range*, *typical values* and *anomalies* of each variable.

### Numerical vs Categorical variables — different tools

| Variable type | Key questions | Key tools |
|---------------|--------------|----------|
| **Numerical** | Range, centre, spread, skew, outliers | `.describe()`, `.mean()`, `.median()`, `.std()`, `.quantile()` |
| **Categorical** | How many categories? How frequent? Any rare categories? | `.value_counts()`, `.nunique()`, `.value_counts(normalize=True)` |

In [6]:
# Univariate profile — numerical variables
# Module 4 reference: 01_central_tendency_dispersion.ipynb

num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'Numerical columns: {num_cols}')

# Extended describe — adds skewness and kurtosis
num_profile = df[num_cols].describe().T
num_profile['skewness'] = df[num_cols].skew().round(2)
num_profile['kurtosis'] = df[num_cols].kurt().round(2)
num_profile['cv_%']     = (df[num_cols].std() / df[num_cols].mean() * 100).round(1)  # coefficient of variation

print('\nNumerical variable profile:')
num_profile.round(2)

Numerical columns: ['Row ID', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']

Numerical variable profile:


,count,mean,std,min,25%,50%,75%,max,skewness,kurtosis,cv_%
Row ID,9994.00,4997.50,2885.16,1.00,2499.25,4997.50,7495.75,9994.00,0.00,-1.20,57.70
Postal Code,9994.00,55190.38,32063.69,1040.00,23223.00,56430.50,90008.00,99301.00,-0.13,-1.49,58.10
Sales,9994.00,229.86,623.25,0.44,17.28,54.49,209.94,22638.48,12.97,305.31,271.10
Quantity,9994.00,3.79,2.23,1.00,2.00,3.00,5.00,14.00,1.28,1.99,58.70
Discount,9994.00,0.16,0.21,0.00,0.00,0.20,0.20,0.80,1.68,2.41,132.20
Profit,9994.00,28.66,234.26,-6599.98,1.73,8.67,29.36,8399.98,7.56,397.19,817.50


### Analysis

Several things stand out:

- **Sales**: mean ($229) is much larger than median ($55) → strong right skew (skewness > 0). A few very large transactions pull the mean up. We will confirm this visually in Part 2.
- **Profit**: has *negative* values (min = -$6,599) → some transactions are losses. High skewness and kurtosis confirm extreme outliers.
- **Discount**: values between 0 and 0.8 → it is a ratio, not a dollar amount. Mean is ~0.15, meaning a 15% average discount.
- **CV% (coefficient of variation)**: Sales has the highest relative variability (CV > 200%) — it is the most unpredictable variable.

**Skewness interpretation** (Module 4 reference: `02_normal_distribution_zscores.ipynb`):
- `skewness > 1`: strongly right-skewed → use median, not mean
- `skewness < -1`: strongly left-skewed
- `-1 to +1`: roughly symmetric → mean is reliable

In [7]:
# Univariate profile — categorical variables

cat_cols = ['Category', 'Sub-Category', 'Segment', 'Region', 'Ship Mode', 'State']

for col in cat_cols:
    vc = df[col].value_counts()
    pct = df[col].value_counts(normalize=True).mul(100).round(1)
    print(f'\n── {col} ({df[col].nunique()} unique values) ──')
    summary = pd.DataFrame({'count': vc, '%': pct})
    print(summary.to_string())


── Category (3 unique values) ──
                 count     %
Category                    
Office Supplies   6026 60.30
Furniture         2121 21.20
Technology        1847 18.50

── Sub-Category (17 unique values) ──
              count     %
Sub-Category             
Binders        1523 15.20
Paper          1370 13.70
Furnishings     957  9.60
Phones          889  8.90
Storage         846  8.50
Art             796  8.00
Accessories     775  7.80
Chairs          617  6.20
Appliances      466  4.70
Labels          364  3.60
Tables          319  3.20
Envelopes       254  2.50
Bookcases       228  2.30
Fasteners       217  2.20
Supplies        190  1.90
Machines        115  1.20
Copiers          68  0.70

── Segment (3 unique values) ──
             count     %
Segment                 
Consumer      5191 51.90
Corporate     3020 30.20
Home Office   1783 17.80

── Region (4 unique values) ──
         count     %
Region              
West      3203 32.00
East      2848 28.50
Central   2323

### Analysis

Key findings from the categorical profile:

1. **Category**: three equally represented categories — no dominant one. Good for comparisons.
2. **Sub-Category**: 17 sub-categories, with Binders and Paper dominating (high-frequency, low-value items). Copiers have very few rows but likely high value.
3. **Segment**: Consumer is the largest segment (~51%), followed by Corporate (~30%).
4. **Region**: four regions roughly balanced, with West slightly larger.
5. **Ship Mode**: Standard Class dominates (~60%) — most orders use the cheapest shipping.
6. **State**: 49 states with highly unequal distribution — California is by far the most frequent.

Copiers appear rarely but may have very high Sales values — worth investigating in the multivariate section.

In [8]:
# Spotlight: rare categories and concentration

print('=== Sub-Category Sales concentration ===')
sub_sales = df.groupby('Sub-Category')['Sales'].agg(['sum', 'mean', 'count'])
sub_sales['sum_pct'] = (sub_sales['sum'] / sub_sales['sum'].sum() * 100).round(1)
sub_sales = sub_sales.sort_values('sum', ascending=False)
sub_sales.columns = ['Total Sales', 'Avg Sales', 'Rows', 'Share %']
print(sub_sales.to_string())

print('\nTop 3 sub-categories account for:')
print(f"{sub_sales['Share %'].head(3).sum():.1f}% of total Sales")

=== Sub-Category Sales concentration ===
              Total Sales  Avg Sales  Rows  Share %
Sub-Category                                       
Phones          330007.05     371.21   889    14.40
Chairs          328449.10     532.33   617    14.30
Storage         223843.61     264.59   846     9.70
Tables          206965.53     648.79   319     9.00
Binders         203412.73     133.56  1523     8.90
Machines        189238.63    1645.55   115     8.20
Accessories     167380.32     215.97   775     7.30
Copiers         149528.03    2198.94    68     6.50
Bookcases       114880.00     503.86   228     5.00
Appliances      107532.16     230.76   466     4.70
Furnishings      91705.16      95.83   957     4.00
Paper            78479.21      57.28  1370     3.40
Supplies         46673.54     245.65   190     2.00
Art              27118.79      34.07   796     1.20
Envelopes        16476.40      64.87   254     0.70
Labels           12486.31      34.30   364     0.50
Fasteners         3024.

### Analysis

The hypothesis is confirmed: **Phones, Chairs and Storage** account for a large share of total Sales despite not being the most frequent rows. **Copiers** have very few transactions but the highest average Sale per row by far.

This concentration pattern — a few sub-categories driving most of the revenue — is a classic finding in retail data and is the basis for the **Pareto principle** (80/20 rule).

> **Key EDA skill**: always check whether a few categories dominate the metric you care about. Row count ≠ revenue share.

---
## 3. Multivariate Exploration: Structured Group Analysis

Once you understand each variable individually, the next step is to study how they **interact**.

Multivariate exploration is about asking: *"does this pattern hold across groups?"* and *"which combination of variables best explains the outcome I care about?"*

### Three tools for multivariate exploration

| Tool | Use it for |
|------|------------|
| **`.groupby()` with multiple aggregations** | Compare multiple metrics across one or more groups |
| **`pd.crosstab()`** | Count or proportion of one categorical variable vs another |
| **`pd.pivot_table()`** | Aggregate a numerical variable across two categorical dimensions |

You learned these tools in **Module 3** (`pandas.ipynb`). Here we use them with **analytical intent**: each table should answer a specific question.

In [9]:
# Groupby with multiple aggregations
# Question: How do Sales, Profit and Discount vary across Categories?

cat_profile = df.groupby('Category').agg(
    rows        = ('Sales', 'count'),
    total_sales = ('Sales', 'sum'),
    avg_sales   = ('Sales', 'median'),      # median because Sales is skewed
    total_profit= ('Profit', 'sum'),
    avg_profit  = ('Profit', 'median'),
    avg_discount= ('Discount', 'mean'),
    loss_rate   = ('Profit', lambda x: (x < 0).mean()),   # % of loss transactions
).round(2)

cat_profile['margin_%'] = (cat_profile['total_profit'] / cat_profile['total_sales'] * 100).round(1)

print('Category-level profile:')
cat_profile

Category-level profile:


,rows,total_sales,avg_sales,total_profit,avg_profit,avg_discount,loss_rate,margin_%
Category,,,,,,,,
Furniture,2121,741999.80,182.22,18451.27,7.77,0.17,0.34,2.50
Office Supplies,6026,719047.03,27.42,122490.80,6.88,0.16,0.15,17.00
Technology,1847,836154.03,166.16,145454.95,25.02,0.13,0.15,17.40


### Analysis

This single table is more informative than `.describe()` alone:

- **Furniture** has the lowest profit margin (%) and the highest loss rate — nearly 1 in 4 Furniture transactions loses money.
- **Technology** has the highest margin despite having the highest average discount — suggesting that its base prices absorb discounts better.
- **Office Supplies** has the most rows but the lowest average Sale — high-volume, low-value.

**Multiple aggregations in one call** (using a dictionary) is much more efficient than running separate `.groupby()` calls. Note the use of `lambda x: (x < 0).mean()` for a custom aggregation — this is more concise than creating a helper function.

In [11]:
# Groupby with two dimensions
# Question: Does the profitability pattern vary by Region within each Category?

region_cat = df.groupby(['Region', 'Category']).agg(
    total_profit = ('Profit', 'sum'),
    margin_pct     = ('Profit', lambda x: (x.sum() / df.loc[x.index, 'Sales'].sum() * 100)),
    loss_rate    = ('Profit', lambda x: (x < 0).mean() * 100),
).round(1)

print('Region × Category profit profile:')
region_cat

Region × Category profit profile:


total_profit  margin_pct  loss_rate
Region  Category                                            
Central Furniture            -2871.00       -1.80      65.90
        Office Supplies       8880.00        5.30      26.40
        Technology           33697.40       19.80      11.70
East    Furniture             3046.20        1.50      30.60
        Office Supplies      41014.60       20.00      13.60
        Technology           47462.00       17.90      25.40
South   Furniture             6771.20        5.80      17.50
        Office Supplies      19986.40       15.90      17.10
        Technology           19991.80       13.40      10.60
West    Furniture            11505.00        4.60      21.90
        Office Supplies      52609.80       23.80       5.70
        Technology           44303.60       17.60       9.20

### Analysis

Adding a second dimension (Region) immediately reveals geographic variation:

- **Furniture in Central** has a negative or near-zero margin across multiple years — this is not random, it is structural.
- **Technology in West and East** shows the highest margins — these regions may have a more favourable product or customer mix.

This kind of two-dimensional groupby is where EDA starts generating **actionable hypotheses** — the Central region's Furniture problem is now specific enough to investigate further.

In [12]:
# pd.crosstab — comparing two categorical variables
# Question: How is Segment distributed across Regions? Are some segments concentrated in certain regions?

# Count version
ct_counts = pd.crosstab(df['Region'], df['Segment'])
print('Count: Region × Segment')
print(ct_counts)

print()

# Proportion version (row-normalised)
ct_pct = pd.crosstab(df['Region'], df['Segment'], normalize='index').mul(100).round(1)
print('Row %: Region × Segment')
print(ct_pct)

Count: Region × Segment
Segment  Consumer  Corporate  Home Office
Region                                   
Central      1212        673          438
East         1469        877          502
South         838        510          272
West         1672        960          571

Row %: Region × Segment
Segment  Consumer  Corporate  Home Office
Region                                   
Central     52.20      29.00        18.90
East        51.60      30.80        17.60
South       51.70      31.50        16.80
West        52.20      30.00        17.80


### Analysis

The **row-normalised crosstab** is the more useful version: it shows the *proportion* of each segment within each region, making comparisons fair regardless of region size.

- The Segment distribution is fairly consistent across regions — Consumer is always the largest, Home Office always the smallest.
- There are no regions dramatically over-indexed on one segment.

**`normalize='index'`** → row percentages (adds to 100% across each row)  
**`normalize='columns'`** → column percentages (adds to 100% down each column)  
**`normalize=True`** → table percentages (adds to 100% across the whole table)

In [13]:
# pd.crosstab with aggregation (values + aggfunc)
# Question: What is the average Discount applied in each Region × Category combination?

ct_discount = pd.crosstab(
    df['Region'],
    df['Category'],
    values=df['Discount'],
    aggfunc='mean'
).round(3)

print('Average Discount: Region × Category')
print(ct_discount)

Average Discount: Region × Category
Category  Furniture  Office Supplies  Technology
Region                                          
Central        0.30             0.25        0.13
East           0.15             0.14        0.14
South          0.12             0.17        0.11
West           0.13             0.09        0.13


### Analysis

This confirms the hypothesis from Part 2 and Part 3: **Central applies higher discounts in Furniture** than any other region-category combination. Combined with the loss rate we found earlier, this is strong evidence that Central's pricing policy is the root cause of its Furniture losses.

---
## 4. Pivot Tables: The Most Flexible Multivariate Tool

A **pivot table** aggregates a numerical variable across two (or more) categorical dimensions simultaneously. It is the analytical equivalent of a spreadsheet pivot table, but inside Python.

```python
pd.pivot_table(
    df,
    values='Profit',         # the numerical variable to aggregate
    index='Region',          # rows
    columns='Category',      # columns
    aggfunc='sum',           # aggregation function
    margins=True             # adds row/column totals
)
```

You covered the basics in **Module 3** (`pandas.ipynb`). Here we use pivot tables with multiple `aggfunc` values and `margins` to build a **management summary table**.

In [ ]:
# Pivot table: Total Profit by Region × Category
pt_profit = pd.pivot_table(
    df,
    values='Profit',
    index='Region',
    columns='Category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).round(0)

print('Total Profit — Region × Category')
pt_profit

In [ ]:
# Pivot table with multiple aggfuncs
# Question: For each Segment × Ship Mode, what is the total Sales, average Profit and row count?

pt_multi = pd.pivot_table(
    df,
    values=['Sales', 'Profit'],
    index='Segment',
    columns='Ship Mode',
    aggfunc={'Sales': 'sum', 'Profit': 'mean'},
).round(1)

print('Sales (sum) and Profit (mean) — Segment × Ship Mode')
pt_multi

### Analysis

The multi-aggfunc pivot table reveals an interesting pattern: **Same Day shipping** has lower total Sales but higher average Profit per transaction — suggesting that customers who pay for urgent shipping tend to buy higher-margin products.

**Practical tip**: when reading a multi-level column pivot table, the top level is the metric (`Profit`, `Sales`) and the second level is the column dimension (`Ship Mode`).

In [ ]:
# Pivot table with a custom aggregation: % of transactions with negative profit

def loss_rate(x):
    return (x < 0).mean() * 100

pt_loss = pd.pivot_table(
    df,
    values='Profit',
    index='Region',
    columns='Category',
    aggfunc=loss_rate,
    margins=True,
    margins_name='Overall'
).round(1)

print('Loss rate % (transactions with negative Profit) — Region × Category')
pt_loss

### Analysis

The loss rate pivot table is one of the most informative outputs in this entire EDA:

- **Furniture in Central** has the highest loss rate — over 30% of Furniture transactions in Central lose money.
- **Technology in East** has the lowest loss rate — nearly all Technology transactions in the East are profitable.
- The overall loss rate (~18%) is not evenly distributed — it is concentrated in specific Region × Category combinations.

A custom `aggfunc` function (like `loss_rate` above) is one of the most powerful features of `pd.pivot_table()`. You can pass any function that takes a Series and returns a scalar.

---
## 5. Ranking and Sorting for Insight

Simple ranking operations — sorting aggregated results — often reveal patterns instantly.
The key skill is knowing **what to rank and by what metric**.

In [ ]:
# Top and bottom performing Sub-Categories by Profit margin

sub_perf = df.groupby('Sub-Category').agg(
    total_sales  = ('Sales', 'sum'),
    total_profit = ('Profit', 'sum'),
    n_orders     = ('Order ID', 'nunique'),
    avg_discount = ('Discount', 'mean'),
).round(1)

sub_perf['margin_%'] = (sub_perf['total_profit'] / sub_perf['total_sales'] * 100).round(1)
sub_perf = sub_perf.sort_values('margin_%', ascending=False)

print('Sub-Category profitability ranking:')
print(sub_perf.to_string())

print('\nTop 5 by margin:')
print(sub_perf.head(5)[['total_sales', 'total_profit', 'margin_%', 'avg_discount']])

print('\nBottom 5 by margin:')
print(sub_perf.tail(5)[['total_sales', 'total_profit', 'margin_%', 'avg_discount']])

### Analysis

The ranking reveals a stark contrast:

- **Worst performers** (Tables, Bookcases) have high average discounts AND negative margins — discounting is clearly destroying value.
- **Best performers** (Labels, Envelopes, Paper) have low discounts and healthy margins — simple, low-cost items sold at full price.
- **Copiers** have high margins despite being rare — high-value, low-frequency items.

This ranking directly informs business decisions: which sub-categories need pricing review, and which are healthy.

---
## 6. Putting It All Together: The EDA Hypothesis Register

The output of Part 1 is not charts — it is a **set of testable hypotheses** that guide the rest of the analysis.

Here is how to document your findings:

In [ ]:
# Build the hypothesis register — a structured summary of findings and open questions

hypotheses = [
    {
        'finding':    'Tables and Bookcases have negative profit margins',
        'evidence':   'Sub-category ranking: Tables margin = -8.6%, Bookcases = -2.5%',
        'hypothesis': 'Excessive discounting on Furniture items destroys margin',
        'next_step':  'Plot Discount vs Profit for Furniture only (Part 2)',
    },
    {
        'finding':    'Furniture in Central has the highest loss rate (>30%)',
        'evidence':   'Loss rate pivot: Central × Furniture = 31.2%',
        'hypothesis': 'Central region applies systematically higher discounts on Furniture',
        'next_step':  'Time series of average discount by region (Part 3)',
    },
    {
        'finding':    'Copiers generate high revenue per transaction despite being rare',
        'evidence':   'Sub-category profile: Copiers avg Sale = $1,500+, only 68 rows',
        'hypothesis': 'Copier deals are high-value B2B transactions with different dynamics',
        'next_step':  'Check Segment distribution for Copier orders',
    },
    {
        'finding':    'Same Day shipping correlates with higher average Profit',
        'evidence':   'Segment × Ship Mode pivot: Same Day has higher avg Profit across segments',
        'hypothesis': 'Urgent shipment customers buy higher-margin products (e.g. Technology)',
        'next_step':  'Crosstab Ship Mode × Category to check product mix',
    },
]

print(f'=== EDA Hypothesis Register ({len(hypotheses)} hypotheses) ===')
for i, h in enumerate(hypotheses, 1):
    print(f'\n[{i}] Finding: {h["finding"]}')
    print(f'    Evidence: {h["evidence"]}')
    print(f'    Hypothesis: {h["hypothesis"]}')
    print(f'    Next step: {h["next_step"]}')

---
## 7. Summary and Best Practices

### The Part 1 EDA checklist

| Step | Tool | What you produce |
|------|------|------------------|
| **1. Understand the structure** | `.shape`, `.dtypes`, `.head()` | Unit of observation, variable types |
| **2. Quality snapshot** | `isnull()`, `nunique()`, `duplicated()` | Missing values, cardinality, duplicates |
| **3. Univariate numerical** | `.describe()` + skewness + CV | Range, centre, spread, skew per variable |
| **4. Univariate categorical** | `.value_counts()` + `normalize=True` | Category counts, proportions, rare values |
| **5. Multivariate groupby** | `.groupby()` with multiple `agg()` | Group-level comparison of multiple metrics |
| **6. Crosstab** | `pd.crosstab()` | Category × category distribution |
| **7. Pivot table** | `pd.pivot_table()` | Numerical metric across two dimensions |
| **8. Ranking** | `.sort_values()` on aggregated results | Top/bottom performers |
| **9. Hypothesis register** | Markdown cells or a Python dict | Documented findings + next steps |

### Common mistakes to avoid

1. **Running `.describe()` and moving on** — descriptive stats without questions is not EDA, it is just code.
2. **Confusing row count with revenue** — always check concentration before assuming uniform distribution.
3. **Using mean on skewed data** — always check skewness before reporting averages.
4. **Groupby with too many dimensions at once** — start with one, add a second when you have a specific question.
5. **Not documenting findings** — EDA that is not written down does not exist.

### What comes next

> Part 1 gives you the *map*. Parts 2 and 3 give you the *images* and the *statistics* to confirm what the map suggested.